# CoherenceBench-IN — Springer LNCS Format

This notebook generates `paper/springer/main_springer.tex` ready for submission to a Springer LNCS venue (e.g., ECIR, ECML, EMNLP workshops via Springer).

**Authors:**
- Sowmya B J — AI & DS, Ramaiah Institute of Technology, `sowmyabj@msrit.edu`
- Jeeth Bhavesh Kataria — AI & DS, Ramaiah Institute of Technology, `jeethkataria9798@gmail.com`

**Document class:** `llncs` (Springer Lecture Notes in Computer Science)

**Run all cells** to regenerate after any section edits.

In [ ]:
# ─── Setup ────────────────────────────────────────────────────────
import json, pathlib, shutil, numpy as np, pandas as pd

BASE         = pathlib.Path('../')
RESULTS_DIR  = BASE / 'results'
PAPER_DIR    = BASE / 'paper'
SP_DIR       = PAPER_DIR / 'springer'
SP_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILES = {
    'Llama-3.2-3B':  'llama3_3b',
    'Qwen2.5-3B':    'qwen25_3b',
    'Mistral-7B':    'mistral_7b',
    'Qwen2.5-7B':    'qwen25_7b',
    'Qwen2.5-14B':   'qwen25_14b',
}
DIMS = ['entity_consistency', 'temporal_coherence', 'causal_chain']

results = {}
for name, fname in MODEL_FILES.items():
    fpath = RESULTS_DIR / f'{fname}_results.jsonl'
    if fpath.exists():
        results[name] = pd.DataFrame([json.loads(l) for l in open(fpath)])

def acc(df, **filt):
    s = df
    for c, v in filt.items():
        s = s[s[c] == v]
    return s['correct'].mean() if len(s) else float('nan')

abstract_txt = (PAPER_DIR / 'abstract.txt').read_text() if (PAPER_DIR / 'abstract.txt').exists() else '% Run 04_paper_writing.ipynb first.'

# Copy shared bib
src_bib = PAPER_DIR / 'references.bib'
if src_bib.exists():
    shutil.copy(src_bib, SP_DIR / 'references.bib')
    print('Copied references.bib → paper/springer/')
else:
    print('⚠️  Run 04_paper_writing.ipynb to generate references.bib first.')

print(f'✅ Setup complete. Loaded {len(results)} models. Springer output → paper/springer/')

In [ ]:
# ─── Tables ───────────────────────────────────────────────────────
# Springer LNCS uses \toprule/\midrule/\bottomrule (booktabs) — preferred

table_1 = r"""\begin{table}[t]
\caption{Accuracy (\%) by model and coherence dimension. Chance = 50\%.}
\label{tab:main}
\centering
\begin{tabular}{lcccc}
\toprule
Model & Entity & Temporal & Causal & Overall \\
\midrule
Random baseline & 50.0 & 50.0 & 50.0 & 50.0 \\
\midrule
"""
for model, df in results.items():
    vals = [f"{acc(df, dimension=d)*100:.1f}" for d in DIMS]
    table_1 += f'{model} & ' + ' & '.join(vals) + f' & {acc(df)*100:.1f} \\\\\n'
table_1 += r"""\bottomrule
\end{tabular}
\end{table}
"""

table_2 = r"""\begin{table}[t]
\caption{Accuracy on INCONSISTENT instances (\%) by corruption distance.}
\label{tab:distance}
\centering
\begin{tabular}{lccc}
\toprule
Model & Near ($<$1K) & Mid (1--3K) & Far ($>$3K) \\
\midrule
"""
for model, df in results.items():
    inc  = df[df['gold'] == 'INCONSISTENT']
    near = f"{acc(inc, distance='near')*100:.1f}"
    mid  = f"{acc(inc, distance='mid')*100:.1f}"
    far  = f"{acc(inc, distance='far')*100:.1f}"
    table_2 += f'{model} & {near} & {mid} & {far} \\\\\n'
table_2 += r"""\bottomrule
\end{tabular}
\end{table}
"""

(SP_DIR / 'table1_main.tex').write_text(table_1)
(SP_DIR / 'table2_distance.tex').write_text(table_2)
print('Saved table1_main.tex and table2_distance.tex')
print(table_1)
print(table_2)

In [ ]:
# ─── Keywords file (Springer requires 5+ keywords) ────────────────
keywords = [
    'discourse coherence',
    'large language models',
    'long-context understanding',
    'entity consistency',
    'temporal coherence',
    'causal reasoning',
    'benchmark',
    'natural language processing',
]
kw_str = ' \\and\n'.join(keywords)
print('Keywords:')
for k in keywords:
    print(f'  {k}')

In [ ]:
# ─── Assemble main_springer.tex ───────────────────────────────────

def load_sec(fname):
    f = PAPER_DIR / fname
    return f.read_text() if f.exists() else f'% TODO: run 04_paper_writing.ipynb to generate {fname}'

# Springer LNCS wraps \paragraph into \subsubsection*
def lncs_adapt(text):
    """Convert \\paragraph{X} to \\noindent\\textbf{X.} for LNCS compatibility."""
    import re
    return re.sub(r'\\paragraph\{([^}]+)\}', r'\\noindent\\textbf{\1.}', text)

springer_tex = r"""\documentclass{llncs}
\usepackage{graphicx}
\usepackage{booktabs}
\usepackage{amsmath}
\usepackage{microtype}
\usepackage{url}
\usepackage[hidelinks]{hyperref}
\usepackage{cite}

\begin{document}

\title{CoherenceBench-IN: A Long-Context Discourse Coherence Benchmark
for Evaluating Large Language Models}

\titlerunning{CoherenceBench-IN}

\author{
  Sowmya B J\inst{1} \and
  Jeeth Bhavesh Kataria\inst{1}
}

\authorrunning{B J and Kataria}

\institute{
  Dept.~of Artificial Intelligence and Data Science,
  Ramaiah Institute of Technology, Bangalore, India \\
  \email{\{sowmyabj, jeethkataria9798\}@msrit.edu}
}

\maketitle

\begin{abstract}
""" + abstract_txt + r"""
\end{abstract}

\keywords{
""" + kw_str + r"""
}

\section{Introduction}
""" + lncs_adapt(load_sec('sec_introduction.tex')) + r"""

\section{Related Work}
""" + lncs_adapt(load_sec('sec_related_work.tex')) + r"""

\section{CoherenceBench-IN}
""" + lncs_adapt(load_sec('sec_benchmark.tex')) + r"""

\section{Experimental Setup}
""" + lncs_adapt(load_sec('sec_experiments.tex')) + r"""

\section{Results}
""" + lncs_adapt(load_sec('sec_results.tex')) + r"""

\section{Analysis and Discussion}
""" + lncs_adapt(load_sec('sec_analysis.tex')) + r"""

\section{Conclusion}
""" + lncs_adapt(load_sec('sec_conclusion.tex')) + r"""

""" + (SP_DIR / 'table1_main.tex').read_text() + r"""
""" + (SP_DIR / 'table2_distance.tex').read_text() + r"""

\begin{figure}[t]
\centering
\includegraphics[width=\textwidth]{../../results/paper_fig2_main_results.pdf}
\caption{Accuracy by model and coherence dimension (dashed line = chance).}
\label{fig:main_results}
\end{figure}

\begin{figure}[t]
\centering
\includegraphics[width=\textwidth]{../../results/paper_fig3_context_length.pdf}
\caption{Accuracy vs.\ context length by dimension.}
\label{fig:ctx_length}
\end{figure}

\begin{figure}[t]
\centering
\includegraphics[width=\textwidth]{../../results/paper_fig4_error_analysis.pdf}
\caption{Error breakdown per model (FP = false positive, FN = false negative).}
\label{fig:errors}
\end{figure}

\bibliographystyle{splncs04}
\bibliography{references}

\end{document}
"""

(SP_DIR / 'main_springer.tex').write_text(springer_tex)
print('✅ Saved: paper/springer/main_springer.tex')
print()
print('To compile:')
print('  cd paper/springer')
print('  pdflatex main_springer.tex && bibtex main_springer && pdflatex main_springer.tex && pdflatex main_springer.tex')

In [ ]:
# ─── Checklist ────────────────────────────────────────────────────
files = list(SP_DIR.glob('*'))
print('SPRINGER FORMAT — OUTPUT FILES')
print('=' * 50)
for f in sorted(files):
    sz = f.stat().st_size
    print(f'  ✅ {f.name:<35} ({sz:,} bytes)')

print()
print('Springer LNCS compliance notes:')
notes = [
    '\\documentclass{llncs}                       ✅',
    '\\author with \\inst{} numbering               ✅',
    '\\authorrunning{} short form                  ✅',
    '\\titlerunning{} short form                   ✅',
    '\\institute{} with \\email{}                   ✅',
    '\\keywords{} block (≥5 keywords)              ✅',
    '\\bibliographystyle{splncs04}                 ✅',
    '\\paragraph → \\noindent\\textbf (LNCS compat) ✅',
    'Figures use \\textwidth                       ✅',
    'Page limit check (max 15 pp LNCS typical)    ⬜ — verify after compile',
]
for n in notes:
    print(f'  {n}')

print()
print('Format comparison:')
print(f'  ACL   → paper/main.tex            (acl package, \\And authors)')
print(f'  IEEE  → paper/ieee/main_ieee.tex  (IEEEtran, 2-col, IEEEkeywords)')
print(f'  LNCS  → paper/springer/main_springer.tex  (llncs, \\inst, splncs04)')